# B1 · KNN on mixed data: which columns to put in the distance

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Build a KNN on mixed data (scaled numerics + one-hot categoricals) with a `ColumnTransformer`, measure in cross-validation what happens when adding an **informative** category (`contract_type`) and a **noisy** one
(`city`), choose the hyperparameters with `GridSearchCV` and compare the result with a logistic regression.

## Dataset

`ds.loans(with_missing=False)` with `ratio = amount/income` (1500 applications, 19% defaulters): stratified 70/30 split. Numerics: `ratio, age, job_tenure, income, amount`. Categoricals: `contract_type` (4 levels, informative)
and `city` (12 levels, unrelated to the outcome).

## Educational purpose

Learn to **decide what goes into the distance**: every added column weighs in the measure of "nearness", so a useless feature is a cost, not a neutral. You will also see an honest result: on this problem KNN does not beat a linear model.

## Task

1. Write `build(numerics, categoricals, k=25)`: `Pipeline` of a `ColumnTransformer` (`StandardScaler` on the numerics, `OneHotEncoder(handle_unknown="ignore")` on the categoricals; `sparse_threshold=0`, and if a list is empty the block is not there) and `KNeighborsClassifier(k, weights="distance")`.
2. With `StratifiedKFold(5, shuffle=True, random_state=0)` on the training set compute the mean AUC of four variants and save it in `cv_auc` (`pd.Series`, index: `numerics`, `+ contract`, `+ city`, `+ both`).
3. Save `harmful_variable` (`"contract"` or `"city"`: the one that, added to the numerics, **lowers** the AUC the most) and `city_loss = cv_auc['numerics'] - cv_auc['+ city']`.
4. `GridSearchCV` (same `cv`, `scoring="roc_auc"`) on the `numerics` variant with `n_neighbors` in `[5, 15, 25, 45, 75, 125]`, `weights` in `["uniform", "distance"]` and `p` in `[1, 2]`: save `best_params` (dictionary of the names without prefix) and `knn_test_auc` (test AUC of the best model). Then compare with `lr_test_auc`: a `LogisticRegression(max_iter=1000)` with the same preprocessing of the numerics and `contract_type`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
NUM = ["ratio", "age", "job_tenure", "income", "amount"]
X, y = df[NUM + ["contract_type", "city"]], df["defaulted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
def build(numerics, categoricals, k=25):
    ...

In [ ]:
cv_auc = pd.Series(dtype=float)
harmful_variable = None
city_loss = None
cv_auc

In [ ]:
best_params = None
knn_test_auc = None
lr_test_auc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`blocks = []; if numerics: blocks.append(("num", StandardScaler(), numerics)); if categoricals: blocks.append(("cat", OneHotEncoder(handle_unknown="ignore"), categoricals))`. Search: `GridSearchCV(build(NUM, []), {"kneighborsclassifier__n_neighbors": [...], "kneighborsclassifier__weights": [...], "kneighborsclassifier__p": [1, 2]}, ...)`; then `roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])`. The columns not needed by a variant stay in `X_tr`: the `ColumnTransformer` with `remainder="drop"` discards them.'):
    check(isinstance(build(NUM, ["city"]), Pipeline) and build(NUM, []).fit(X_tr, y_tr).predict(X_te).shape == (len(X_te),), "build returns a Pipeline that also works without categoricals", "the 'cat' block must not be created if the list is empty")
    check(list(cv_auc.index) == ["numerics", "+ contract", "+ city", "+ both"] and 0.72 < cv_auc["numerics"] < 0.82, "four variants; AUC of the numerics between 0.72 and 0.82", "index: numerics, + contract, + city, + both")
    check(harmful_variable == "city" and close(city_loss, cv_auc["numerics"] - cv_auc["+ city"]) and city_loss > 0.005, "the city (noise) lowers the AUC by over 0.005", "compare + contract and + city")
    check(abs(cv_auc["+ contract"] - cv_auc["numerics"]) < 0.02 and cv_auc["+ both"] < cv_auc["+ contract"], "the contract does not help the KNN (difference below 0.02) and adding the city too makes it worse", "read the series cv_auc")
    check(set(best_params) == {"n_neighbors", "weights", "p"} and best_params["n_neighbors"] >= 25, "the search prefers a large k (25 or more): the signal is weak and noisy", "keys without the prefix 'kneighborsclassifier__'")
    check(knn_test_auc > 0.7 and lr_test_auc > knn_test_auc - 0.005, "the logistic is not beaten by the optimized KNN (or nearly tied)", "compare knn_test_auc and lr_test_auc on the test set")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_knn_on_mixed_data.ipynb`. Open it after you pass the check, or after you have really tried.